In [1]:
# pip install pandas numpy matplotlib openpyxl

## DATASET OVERVIEW

In [2]:
from pathlib import Path
import pandas as pd
import gc
BASE_DIR = Path("..").resolve()          # app/
DATA_DIR = BASE_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
OUTPUTS_DIR = DATA_DIR / "outputs"


In [3]:

cols_needed = [
    "Anon Student Id",
    "Problem Name",
    "Step Name",
    "KC(Default)",
    "Correct First Attempt",
    "Incorrects",
    "Hints",
    "Corrects",
    "Step Start Time",
    "Step End Time",
]

cols_06 = cols_needed.copy()
cols_08 = [
    "Anon Student Id",
    "Problem Name",
    "Step Name",
    "KC(SubSkills)",
    "Correct First Attempt",
    "Incorrects",
    "Hints",
    "Corrects",
    "Step Start Time",
    "Step End Time",
]

dtype_06 = {
    "Anon Student Id": "category",
    "Problem Name": "category",
    "Step Name": "category",
    "KC(Default)": "category",
    "Correct First Attempt": "Int8",
    "Incorrects": "Int16",
    "Hints": "Int16",
    "Corrects": "Int16",
}

dtype_08 = {
    "Anon Student Id": "category",
    "Problem Name": "category",
    "Step Name": "category",
    "KC(SubSkills)": "category",
    "Correct First Attempt": "Int8",
    "Incorrects": "Int16",
    "Hints": "Int16",
    "Corrects": "Int16",
}

def load_06(path):
    return pd.read_csv(path, sep="\t", usecols=cols_06, dtype=dtype_06)

def load_08(path):
    df = pd.read_csv(path, sep="\t", usecols=cols_08, dtype=dtype_08)
    return df.rename(columns={
        "KC(SubSkills)": "KC(Default)",
    })

train_06 = load_06(RAW_DIR / "algebra_2006_2007_train.txt")
train_08 = load_08(RAW_DIR / "algebra_2008_2009_train.txt")

_df_train = pd.concat([train_06, train_08], ignore_index=True)

del train_06, train_08
gc.collect()

print("Train shape:", _df_train.shape)
print("Columns:", _df_train.columns.tolist())
print("Students:", _df_train["Anon Student Id"].nunique())
print("Problems:", _df_train["Problem Name"].nunique())
print("Steps:", _df_train["Step Name"].nunique())
print("KCs:", _df_train["KC(Default)"].nunique())
print(_df_train.head())


FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\Melany Nuria\\Desktop\\TFG\\adaptive_bayesian_its\\backend_fastAPI\\app\\notebooks\\data\\raw\\algebra_2006_2007_train.txt'

## Helpers

In [ ]:
def convert_time_columns(df):
    time_cols = [
        "Step Start Time",
        "Step End Time"
    ]
    for col in time_cols:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")
    return df


def convert_numeric_columns(df):
    numeric_cols = [
        "Correct First Attempt",
        "Incorrects",
        "Hints",
        "Corrects",
    ]
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    return df

# =========================================================
# KC MAPPING
# =========================================================
def map_kc(kc):
    if pd.isna(kc):
        return None

    kc_lower = str(kc).lower().strip()

    exclude_patterns = [
        "skillrule: variable in denominator;",
        "clt-r-den",
        "clt-r-num",
        "combine-like-terms-r-sp",
    ]
    if any(p in kc_lower for p in exclude_patterns):
        return "Other"
    

    # 1. Expand / Eliminate Parentheses
    if (
        "skillrule: eliminate parens;" in kc_lower
        or "skillrule: calculate eliminate parens;" in kc_lower
        or "skillrule: do eliminate parens - whole;" in kc_lower
        or "skillrule: extract to consolidate vars;" in kc_lower
        or "skillrule: select eliminate parens" in kc_lower

    ):
        return "Expand / Eliminate Parentheses"

    # 2. Combine Like Terms
    if (
        "skillrule: combine like terms, no var;" in kc_lower
        or "skillrule: do combine terms - whole;" in kc_lower
        or "skillrule: consolidate vars, no coeff;" in kc_lower
        or "skillrule: consolidate vars with coeff;" in kc_lower
        or "skillrule: consolidate vars, any;" in kc_lower
        or "skillrule: add/subtract" in kc_lower
    ):
        return "Combine Like Terms"

    # Optional: only keep a restricted CLT fallback
    if (
        "clt" in kc_lower
        and "parens" not in kc_lower
        and "consolidate vars" not in kc_lower
        and "r-den" not in kc_lower
        and "r-num" not in kc_lower
    ):
        return "Combine Like Terms"

    # 3. Move Constants Across the Equation
    if (
        "skillrule: remove constant;" in kc_lower
        or "skillrule: ax+b=c, negative;" in kc_lower
        or "skillrule: isolate positive;" in kc_lower
        or "skillrule: isolate negative;" in kc_lower
    ):
        return "Move Constants Across the Equation"

    # 4. Remove Coefficient
    if (
        "skillrule: remove coefficient;" in kc_lower
        or "skillrule: remove positive coefficient;" in kc_lower
        or "skillrule: remove negative coefficient;" in kc_lower
        or "skillrule: multiply/divide;" in kc_lower
        or "[solveroperation multiply]" in kc_lower
        or "[solveroperation divide]" in kc_lower
        or "solveroperation mt" in kc_lower
        or "solveroperation rf" in kc_lower
        or "rf right" in kc_lower
        or "rf left" in kc_lower
    ):
        return "Remove Coefficient"

    # 5. Normalize Negative Variable / Sign
    if (
        "skillrule: make variable positive;" in kc_lower
        or "skillrule: calculate negative coefficient;" in kc_lower
    ):
        return "Normalize Negative Variable / Sign"

    return "Other"

def is_linear_equation_step(step_name):
    """
    Keep only rows that look like simple first-degree linear equation steps.
    """
    if pd.isna(step_name):
        return False

    step = str(step_name).lower().strip()

    if "=" not in step:
        return False

    # Off-domain / non-linear / obviously irrelevant patterns
    banned_patterns = [
        "sqrt", "^", "hypotenuse", "slope", "geometry",
        "probability", "median", "mean", "mode", "ratio",
        "scientific notation", "axis", "plot", "graph",
        "quadrat", "exponent", "compare", "action",
        "unspecified", "unknown", "any form", "entering a",
        "sin", "cos", "tan", "log", "ln", "abs"
    ]

    if any(p in step for p in banned_patterns):
        return False

    return True


def preprocess_kc(
    df,
    remove_other=False,
    drop_original_kc_cols=False,
    filter_irrelevant=False
):
    df = df.dropna(subset=[
        "Anon Student Id",
        "KC(Default)",
        "Correct First Attempt",
        "Step Start Time"
    ]).copy()

    df = df[df["Step Name"].apply(is_linear_equation_step)].copy()


    # Optional domain filtering
    if filter_irrelevant:
        irrelevant_patterns = [
            "action", "compare", "axis", "-row",
            "unspecified", "unknown", "any form", "entering a",
            "plot", "graph", "probability", "median", "mean",
            "mode", "rate", "ratio", "scientific notation", "slope",
            "square root", "hypotenuse", "geometry", "quadrat", "exponent"
        ]

        mask = ~df["KC(Default)"].astype(str).str.lower().apply(
            lambda x: any(pattern in x for pattern in irrelevant_patterns)
        )
        df = df[mask]

    # Split multi-KC rows
    df["KC"] = df["KC(Default)"].astype(str).str.split("~~")

    # One row per KC
    df = df.explode("KC")
    input_rows = df.shape[0]

    # Clean KC values
    df["KC"] = df["KC"].astype(str).str.strip()
    df = df[df["KC"] != ""]
    df = df[df["KC"].str.lower() != "nan"]


    # Map to reduced KC taxonomy
    df["skill_name"] = df["KC"].apply(map_kc)

    if remove_other:
        df = df[df["skill_name"] != "Other"]

    if drop_original_kc_cols:
        cols_to_drop = [col for col in ["KC(Default)", "KC"] if col in df.columns]
        df = df.drop(columns=cols_to_drop)

    df = df.reset_index(drop=True)
    output_rows = df.shape[0]

    print(f"Dataset input rows: {input_rows}")
    print(f"Dataset output rows: {output_rows}\n")

    return df


## KC election

Some steps have multiple KCs associated with them; when this occurs, they are separated by ~~. In this section, we create one row per KC and then classify them to obtain a reduced and interpretable set of KCs related to first-degree linear equations:
1. Expand / Eliminate Parentheses: removing or expanding parentheses through the distributive property or equivalent transformations.
2. Move Constants Across the Equation: using inverse addition or subtraction to transfer constant terms from one side of the equation to the other.
3. Combine Like Terms: simplifying expressions by combining terms of the same type on the same side of the equation.
4. Remove Coefficient: eliminating the coefficient of the variable, typically through multiplication or division, in order to isolate it.
5. Normalize Negative Variable / Sign: handling negative signs so that the variable is rewritten with a positive coefficient.

In [ ]:
# df = _df_train[cols_needed].copy()

# df = preprocess_kc(df, remove_other=True, filter_irrelevant=True)

# # 4. BUILD MAIN KC -> PROBLEM -> STEP TABLE
# kc_problem_step = (
#     df.groupby(["KC", "skill_name", "Problem Name", "Step Name"])
#       .agg(
#           appearances=("KC", "size"),
#           students=("Anon Student Id", "nunique"),
#           avg_correct=("Correct First Attempt", "mean")
#       )
#       .reset_index()
#       .sort_values(["skill_name", "appearances"], ascending=[True, False])
# )

# # 5. UNIQUE KCs WITH COUNTS
# kc_summary = (
#     df.groupby("KC")
#       .agg(
#           total_appearances=("KC", "size"),
#           n_problems=("Problem Name", "nunique"),
#           n_steps=("Step Name", "nunique"),
#           n_students=("Anon Student Id", "nunique"),
#           avg_correct=("Correct First Attempt", "mean")
#       )
#       .reset_index()
#       .sort_values("total_appearances", ascending=False)
# )

# # 6. EXPORT FOR MANUAL REVIEW
# kc_problem_step.to_csv(RAW_DIR / "kc_problem_step_mapping.csv", index=False)
# kc_summary.to_csv(RAW_DIR / "kc_summary.csv", index=False)


## Dataset Processing

Final dataset with only relevant KC columns

In [ ]:
df_train = convert_time_columns(_df_train)
del _df_train
gc.collect()
# df_test = convert_time_columns(df_test)

df_train = convert_numeric_columns(df_train)
# df_test = convert_numeric_columns(df_test)

df_train = preprocess_kc(df_train,remove_other=True,drop_original_kc_cols=True,filter_irrelevant=True)
# df_test = preprocess_kc(df_test,remove_other=True,drop_original_kc_cols=True,filter_irrelevant=True)

# Sort in true chronological order per student
df_train = df_train.sort_values(
    by=["Anon Student Id", "Step Start Time", "Problem Name", "Step Name"]
).reset_index(drop=True)

# df_test = df_test.sort_values(
#     by=["Anon Student Id", "Step Start Time", "Problem Name", "Step Name"]
# ).reset_index(drop=True)

# REMOVE DUPLICATES
# Include Step Start Time to preserve separate interactions
subset_cols = [
    "Anon Student Id",
    "Problem Name",
    "Step Name",
    "skill_name",
    "Step Start Time"
]

duplicates_train = df_train.duplicated(subset=subset_cols)
# duplicates_test = df_test.duplicated(subset=subset_cols)

n_duplicates_train = duplicates_train.sum()
# n_duplicates_test = duplicates_test.sum()

percentage_train = (n_duplicates_train / len(df_train)) * 100 if len(df_train) > 0 else 0
# percentage_test = (n_duplicates_test / len(df_test)) * 100 if len(df_test) > 0 else 0

print(f"Train duplicated interactions: {n_duplicates_train}")
print(f"Train duplicate percentage: {percentage_train:.2f}%")

# print(f"Test duplicated interactions: {n_duplicates_test}")
# print(f"Test duplicate percentage: {percentage_test:.2f}%")

df_train = df_train.drop_duplicates(subset=subset_cols).reset_index(drop=True)
# df_test = df_test.drop_duplicates(subset=subset_cols).reset_index(drop=True)

print("Remaining train duplicates:", df_train.duplicated(subset=subset_cols).sum())
# print("Remaining test duplicates:", df_test.duplicated(subset=subset_cols).sum())

# =========================================================
# FINAL DATASET INFO
# =========================================================
print("\nFINAL TRAIN DATASET")
print("Shape:", df_train.shape)
print("Columns:", df_train.columns.to_list())
print("Number of students:", df_train["Anon Student Id"].nunique())
print("Number of problems:", df_train["Problem Name"].nunique())
print("Number of steps:", df_train["Step Name"].nunique())
print("Number of reduced KCs:", df_train["skill_name"].nunique())

print("\nReduced KC counts:")
print(df_train["skill_name"].value_counts())


Dataset input rows: 2245300
Dataset output rows: 1645700

Train duplicated interactions: 537008
Train duplicate percentage: 32.63%
Remaining train duplicates: 0

FINAL TRAIN DATASET
Shape: (1108692, 10)
Columns: ['Anon Student Id', 'Problem Name', 'Step Name', 'Step Start Time', 'Step End Time', 'Correct First Attempt', 'Incorrects', 'Hints', 'Corrects', 'skill_name']
Number of students: 3281
Number of problems: 235101
Number of steps: 616254
Number of reduced KCs: 5

Reduced KC counts:
skill_name
Remove Coefficient                    352053
Combine Like Terms                    319401
Move Constants Across the Equation    273497
Expand / Eliminate Parentheses        147066
Normalize Negative Variable / Sign     16675
Name: count, dtype: int64


## Final cleaned files

In [ ]:
train_file = PROCESSED_DIR / "df_train_kc_cleaned_2006-2009_v.0.csv"
# test_file = PROCESSED_DIR / "df_test_kc_cleaned.csv"

df_train.to_csv(train_file, index=False)
# df_test.to_csv(test_file, index=False)

print(f"Saved: {train_file}")

Saved: /mnt/c/Users/Melany Nuria/Desktop/TFG/adaptive_bayesian_its/backend_fastAPI/app/data/processed/df_train_kc_cleaned_2006-2009_2.csv


In [16]:
def export_kc_with_context(
    df,
    output_file="kc_with_prev_next_context.xlsx",
    target_skill=None,
    student_col="Anon Student Id",
    problem_col="Problem Name",
    time_cols=("Step Start Time", "Step End Time"),
    step_col="Step Name",
    kc_col="skill_name"
):
    data = df.copy()

    # -------------------------------------------------
    # 1. Ensure time columns are datetime
    # -------------------------------------------------
    for col in time_cols:
        if col in data.columns:
            data[col] = pd.to_datetime(data[col], errors="coerce")

    # -------------------------------------------------
    # 2. Sort within each student-problem interaction
    # -------------------------------------------------
    sort_cols = [c for c in [student_col, problem_col, *time_cols, step_col] if c in data.columns]
    data = data.sort_values(sort_cols).reset_index(drop=True)

    # -------------------------------------------------
    # 3. Add row number inside each interaction
    # -------------------------------------------------
    group_cols = [student_col, problem_col]
    data["step_index_in_problem"] = data.groupby(group_cols).cumcount()

    # -------------------------------------------------
    # 4. Previous / next step context
    # -------------------------------------------------
    data["prev_step_name"] = data.groupby(group_cols)[step_col].shift(1)
    data["next_step_name"] = data.groupby(group_cols)[step_col].shift(-1)

    if "KC(Default)" in data.columns:
        data["prev_kc_default"] = data.groupby(group_cols)["KC(Default)"].shift(1)
        data["next_kc_default"] = data.groupby(group_cols)["KC(Default)"].shift(-1)

    if "KC" in data.columns:
        data["prev_kc"] = data.groupby(group_cols)["KC"].shift(1)
        data["next_kc"] = data.groupby(group_cols)["KC"].shift(-1)

    if kc_col in data.columns:
        data["prev_skill_name"] = data.groupby(group_cols)[kc_col].shift(1)
        data["next_skill_name"] = data.groupby(group_cols)[kc_col].shift(-1)

    if "Correct First Attempt" in data.columns:
        data["prev_correct_first_attempt"] = data.groupby(group_cols)["Correct First Attempt"].shift(1)
        data["next_correct_first_attempt"] = data.groupby(group_cols)["Correct First Attempt"].shift(-1)

    # -------------------------------------------------
    # 5. Optional: keep only one KC
    # -------------------------------------------------
    if target_skill is not None and kc_col in data.columns:
        data = data[data[kc_col] == target_skill].copy()

    # -------------------------------------------------
    # 6. Choose useful columns first
    # -------------------------------------------------
    preferred_cols = [
        student_col,
        problem_col,
        "step_index_in_problem",
        "prev_step_name",
        step_col,
        "next_step_name",
        "prev_kc_default",
        "KC(Default)",
        "next_kc_default",
        "prev_kc",
        "KC",
        "next_kc",
        "prev_skill_name",
        kc_col,
        "next_skill_name",
        "prev_correct_first_attempt",
        "Correct First Attempt",
        "next_correct_first_attempt",
        "Incorrects",
        "Hints",
        "Corrects",
        "Step Start Time",
        "Step End Time",
    ]

    final_cols = [c for c in preferred_cols if c in data.columns] + [
        c for c in data.columns if c not in preferred_cols
    ]

    data = data[final_cols]

    # -------------------------------------------------
    # 7. Export
    # -------------------------------------------------
    data.to_excel(output_file, index=False)

    print(f"Saved: {output_file}")
    print(f"Rows exported: {len(data)}")

    return data

expand_context = export_kc_with_context(
    df_train,
    output_file="expand_eliminate_parentheses_with_context.xlsx",
    target_skill="Expand / Eliminate Parentheses"
)

Saved: expand_eliminate_parentheses_with_context.xlsx
Rows exported: 147066


In [ ]:
# # One full interaction per exercise (Problem Name) from the final dataset

# # Recommended columns to define one interaction:
# # - Problem Name
# # - Anon Student Id
# # Keep all rows/steps for the first occurrence of each exercise.

# def extract_one_interaction_per_exercise(
#     df,
#     exercise_col="Problem Name",
#     student_col="Anon Student Id",
#     time_cols=("Step Start Time", "First Transaction Time")
# ):
#     df_sample = df.copy()

#     # Sort so the chosen interaction is reproducible
#     sort_cols = [c for c in [exercise_col, student_col, *time_cols] if c in df_sample.columns]
#     df_sample = df_sample.sort_values(sort_cols).reset_index(drop=True)

#     # Identify the first student occurrence for each exercise
#     interaction_keys = [exercise_col, student_col]
#     first_occurrences = (
#         df_sample[interaction_keys]
#         .drop_duplicates()
#         .drop_duplicates(subset=[exercise_col], keep="first")
#         .reset_index(drop=True)
#     )

#     # Keep all steps belonging to those selected interactions
#     sample = df_sample.merge(first_occurrences, on=interaction_keys, how="inner")

#     return sample.reset_index(drop=True)


# # Apply it to your final dataset
# kc_problem_step_mapping = extract_one_interaction_per_exercise(df_train)

# print("Rows:", len(kc_problem_step_mapping))
# print("Unique exercises:", kc_problem_step_mapping["Problem Name"].nunique())

# # Optional: save it
# kc_problem_step_mapping.to_excel("kc_problem_step_mapping.xlsx", index=False)

Rows: 635719
Unique exercises: 235101


Notas: 
- cata studiante tiene su P(L) prob. of learning 
- este dataset nos ayuda a obtener las probabilidades Pl0, pt, pg, ps para cada KC 
- despues de cada ejercicio realizadp la P(L) del estudiante se actualiza